# 🏆 Comment Category Prediction Challenge —


## Objective of the project


The objective of this project is to develop a machine learning model that determines the appropriate category for each comment by leveraging text features, user interaction signals, and engineered attributes.

##  Importing the required libraries and settings


In [ ]:

import re, os, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, ConfusionMatrixDisplay

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import xgboost as xgb
import lightgbm as lgb



SEED = 42   # done to fix the randomness ( results remain consistent across run)


print("Imports OK")

##  Loading of the data

In [ ]:
train = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/test.csv')
sample_submission = pd.read_csv('/kaggle/input/comment-category-prediction-challenge/Sample.csv')


In [ ]:
print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"Sub   : {sample_submission.shape}")


In [ ]:
train.head(3)

In [ ]:
test.head(3)

#  Identifying the different columns of datasets

In [ ]:
train.info()

In [ ]:
test.info()

# Descriptive statistics of numerical columns

In [ ]:
train.describe()

- as we can see the count column - 198000 that means out dataset has no missing values which means high quality dataset.
- most of the features are highly skewed with many 0's.(median(50%) = 0)
- Label distribution: max = 3 , so multi class classification 

In [ ]:
test.describe()

#  Identify and handle duplicated values 

In [ ]:
train.duplicated().sum()

In [ ]:
test.duplicated().sum()

Hence, we can see no duplicated value are present in the dataset, so we can move on. 

# Identify missing values: 

In [ ]:
train.isna().sum()

In [ ]:
test.isna().sum()

# EDA ( Exploratory data analysis) 

Each plot is designed to justify a specific engineering decision downstream.

In [ ]:
# Label Distribution
import seaborn as sns

sns.countplot(x='label', data=train)
plt.title("Label Distribution")
plt.xlabel("Label")
plt.ylabel("Count")
plt.show()



as you can se,  Dataset is highly imbalanced
Class 0 has the highest count (dominates the dataset)
Class 3 has very few samples

here, the risk in model training is it may become biases toward class 0. 

thats why: 
"Due to class imbalance, we used Macro F1 score to give equal importance to all classes instead of accuracy.

- Macro F1 gives equal importance to all classes, even rare ones.

##  Comment Length vs Label



In [ ]:
# Comment Length vs Label

train['comment_length'] = train['comment'].fillna('').apply(len)

sns.boxplot(x='label', y='comment_length', data=train)
plt.title("Comment Length vs Label")
plt.xlabel("Label")
plt.ylabel("Comment Length")
plt.show()

Comment length varies across classes Labels 1 and 2 → generally longer comments Labels 0 and 3 → relatively shorter comments

The boxes overlap a lot No clear separation between classes

Comment length can be a useful supporting feature but not a strong standalone predictor.

##  Emoticon Usage vs Label


In [ ]:
# Emoticon usage vs Label

emo_cols = ['emoticon_1', 'emoticon_2', 'emoticon_3']

emo_mean = train.groupby('label')[emo_cols].mean()

emo_mean.plot(kind='bar')
plt.title("Mean Emoticon Usage per Label")
plt.xlabel("Label")
plt.ylabel("Mean Value")
plt.xticks(rotation=0)
plt.show()

Emoticon_1 is most used across all labels
"Different labels show different emoticon usage patterns."

We observed slight variation in emoticon usage across labels, but differences are small, so they act as supporting features rather than primary predictors.

##  Feature Correlation Analysis


- High correlation mean ---> strong relationship between two variables
- low correlation means _-> weak or no relationship

In [ ]:
# Numeric Feature Correlation

num_cols = ['upvote','downvote','emoticon_1','emoticon_2','emoticon_3','if_1','if_2']

corr = train[num_cols].corr()

sns.heatmap(corr, annot=True)
plt.title("Feature Correlation")
plt.show()

1. Moderate correlation between upvotes and downvotes means Posts with more engagement tend to get both upvotes and downvotes.
2. Emoticon_3 and downvotesrelation : (emoticon_3) may be associated with more -ve reaction.
3. Most features have values close to 0(low correlation) 
4. if_1 and if_2 are almost useless (very low correlation)

+ correlation -> both moving in same direction
  - corr --> both movng in diff dirr. no reationship
 

Most features show low correlation, indicating independence, with only moderate relationships like upvotes–downvotes and emoticon_3–downvotes.

#  Preprocessing

##  filling the null values for numerical col and also categorical column



In [ ]:
# ──  numeric coercion ─────────────────────────────────────────────────────
num_cols = ['upvote','downvote','emoticon_1','emoticon_2','emoticon_3','if_1','if_2']
cat_cols = ['race','religion','gender','disability']

for df in [train, test]:
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)  # = coerce - convert invalid values into NaN insted throw error
    df['comment'] = df['comment'].fillna('')   # fill with empty sting
    for col in cat_cols:
        df[col] = df[col].fillna('none').astype(str).str.lower().str.strip()

print("Basic cleaning done.")

##  Text Cleaning


- 1st line ; convert the input into string and also lowercase .
- 2nd line ; to remove the url like http, www form the text
- 3rd line ; to remove the repeted char to max of 2. 
- 4th line ; remove special char and keep letter and digits. 
- 5th line; to remove the extra spaces from front and end. 
- return clean text. 

In [ ]:
# ── text cleaning ────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)        # remove URLs
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)           # loooove → loove (cap at 2)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)              # keep digits 
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['clean_comment'] = train['comment'].apply(clean_text)
test['clean_comment']  = test['comment'].apply(clean_text)

# Sanity check — show diff for a few rows
sample = train['comment'].head(3).tolist()
for orig in sample:
    print(f"  ORIG: {orig[:80]}")
    print(f"  CLEAN: {clean_text(orig)[:80]}")
    print()

# Feature Engineering

1. High net_votes → positive reaction
2. High downvotes → controversial or negative
3. total_votes --> capture high engagement
4. comment length : more informative or more emotional
5. vote_ratio --> positive vs negative balance.( to avoid division by 0) 

6. log_upvote and log_downvote -->  To reduces skewness and stabilizes variance
7. upvote rate --> measures proportion of positive feedback.
8. cotroversy score --> high value(mixed opinion) and low value - clear sentiment. 

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ──new feature from Original features  ──────────────────────────────────────────────
    df['net_votes']      = df['upvote'] - df['downvote']
    df['total_votes']    = df['upvote'] + df['downvote']
    df['vote_ratio']     = df['upvote'] / (df['downvote'] + 1)
    df['comment_length'] = df['comment'].apply(len)
    df['word_count']     = df['comment'].apply(lambda x: len(x.split()))

    # ──  log-scaled vote features (stabilise skewed distributions) ────────
    df['log_upvote']   = np.log1p(df['upvote'])
    df['log_downvote'] = np.log1p(df['downvote'])

    # ──  engagement quality signals ───────────────────────────────────────
    df['upvote_rate']        = df['upvote'] / (df['total_votes'] + 1)
    df['controversy_score']  = (
        df[['upvote','downvote']].min(axis=1) /
        (df[['upvote','downvote']].max(axis=1) + 1)
    )

    # ── NEW: comment structure signals (computed on RAW comment, before cleaning) ──
    raw = df['comment'].fillna('')
    df['has_question']    = raw.str.contains(r'\?').astype(int) # comment contain question 
    df['has_exclamation'] = raw.str.contains(r'!').astype(int)   # it may have excitement or strong emotion. 
    df['caps_ratio']      = raw.apply(
        lambda x: sum(1 for c in x if c.isupper()) / (len(x) + 1)   # uppercase shows emphasis on the words in comment. 
    )

    # ──(on clean comment) ────────────────────
    clean = df['clean_comment'].fillna('')
    words = clean.str.split()
    df['avg_word_len']      = words.apply(                       # captures complexity of language
        lambda ws: np.mean([len(w) for w in ws]) if ws else 0
    )
    df['unique_word_ratio'] = words.apply(                       #  it captures vocabulary diversity. if lower then repetitive text .or different wording is used. 
        lambda ws: len(set(ws)) / (len(ws) + 1) if ws else 0
    )

    return df

train = add_features(train)
test  = add_features(test)

print(f"Features added. Train: {train.shape}  Test: {test.shape}")

previously we have : Train (198000,15) Test: (102000,14)

In [ ]:
# ── Verify no NaN introduced by feature engineering ──────────────────────────
new_feats = ['log_upvote','log_downvote','upvote_rate','controversy_score',
             'has_question','has_exclamation','caps_ratio',
             'avg_word_len','unique_word_ratio']

print("NaN check on new features:")
print(train[new_feats].isnull().sum())


##  Column Definitions

In [ ]:
text_feature = 'clean_comment'

numeric_features = [
    # Original
    'upvote','downvote','net_votes','total_votes','vote_ratio',
    'comment_length','word_count',
    'emoticon_1','emoticon_2','emoticon_3','if_1','if_2',
    # New
    'log_upvote','log_downvote','upvote_rate','controversy_score',
    'has_question','has_exclamation','caps_ratio',
    'avg_word_len','unique_word_ratio',
]

categorical_features = ['race','religion','gender','disability']

# Confirm all columns exist
for c in numeric_features + categorical_features + [text_feature]:
    for df, name in [(train,'train'),(test,'test')]:
        if c not in df.columns:
            print(f" MISSING in {name}: {c}")
print("Column check complete.")

## Train-Validation Split

Splitting the dataset into training and validation sets while preserving the label distribution by stratify = y. 

- 80% training, 20% validation
- Validation set is used to evaluate model performance on unseen data
- stratify = y, it ensure same label distribution in train and validation. also as out data set is imbalence , so we have to ensure all classes are proportionally represented in both train and validation set 
-  random_state - same split every time the code runs. 

In [ ]:
from sklearn.model_selection import train_test_split


X = train.drop(columns=['label'])
y = train['label']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size    = 0.2,
    stratify     = y,   # to maintain lebel distribution in train , val
    random_state = SEED,
)

print(f"Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}")
print(f"Val label distribution:\n{y_val.value_counts().sort_index()}")

##  Scaling and encoding (TF-IDF Vectorization, standard scaler and OneHotEncoding) 


- Text data is converted into numerical features using TF-IDF
-  ngram_range=(1,2) : Captures both individual words and short phrases
- stop_words="english": Removes common words like ‘the’, ‘is’
- min_df=2: Removes rare words that appear only once (noise reduction).”
- sublinear_tf=True: Applies log scaling to term frequency to reduce dominance of frequent words.

1. Character-level TF-IDF captures sub-word patterns (like spelling variations, prefixes, suffixes), which word-level TF-IDF misses.

In [ ]:
# ── TF-IDF: word n-grams ──────────────────────────────────────────────────────

word_tfidf = TfidfVectorizer(
    max_features = 20000,    # limit to the top 20,000 most imp words 
    ngram_range  = (1, 2),     # both unigram and bigrams (single or double words) ,(for better context)
    stop_words   = 'english',   # removes word like this, is, and ( reduce noice)
    sublinear_tf = True,          # applies log scaling from preventing word dominating. 
    min_df       = 2,             # ignore words appear in fewer than 2
    dtype        = np.float32,    # store values in 32 bit float instead of 64 bits.
)

# ── TF-IDF: character n-grams ────────────────────────────────────────────────

char_tfidf = TfidfVectorizer(
    analyzer     = 'char_wb',   # char level ngram
    ngram_range  = (3, 5),       # capture pattern inside words upto 3 to 5 len
    max_features = 8000,           # limit upto 8000 char
    sublinear_tf = True,          # applies log scaling to freq.
    min_df       = 2,              # remove rare char patterns
    dtype        = np.float32,
)

preprocessor = ColumnTransformer(
    transformers=[
        ('word_tfidf', word_tfidf,                           text_feature),
        ('char_tfidf', char_tfidf,                           text_feature),
        ('num',        StandardScaler(),                     numeric_features),
        ('cat',        OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ],
    n_jobs = -1,
)


##  Model Training

Helper functions are defined for reuse in the pipeline.
To simplify code and avoid repetition.

## Preprocess once for all. 

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)
X_val_t   = preprocessor.transform(X_val)
test_t    = preprocessor.transform(test)

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline

##  Logistic Regression Model



Logistic Regression is a supervised learning algorithm used for classification problems. It predicts the probability of a class using the sigmoid function
- Used for binary and multiclass classification.
- pipeline used to combine multiple steps ( preprocessing , model training at once ) 

In [ ]:
# # Pipeline
# log_pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('classifier', LogisticRegression(
#         max_iter=3000,          # max iter to ensure convergence.
#         solver='saga',          # efficient for large dataset 
#         multi_class='multinomial', # handle multi class classification
#         class_weight='balanced',    # for adjusting weight as this is imbalanced dataset
#         random_state=SEED,
#         n_jobs=-1                   # use all avail. cpu cores for computation
#     ))
# ])

# # Param grid
# log_param_grid = {
#     'classifier__C': [0.1, 1, 4, 10]     # tuning the regularization parameter C
# }

# # Tuning
# log_grid = GridSearchCV(
#     log_pipeline,
#     log_param_grid,
#     scoring='f1_macro',     # f1 marco for dataset imbalenced.
#     cv=3,                     # 3 fold cross validation 
#     n_jobs=-1,                  # parallel processing for faster tuning
#     verbose=2      # it control level of output/
# )

# log_grid.fit(X_train, y_train)

# #  Print best params
# print("\n===== Logistic Regression =====")
# print("Best Params:", log_grid.best_params_)
# print("Best CV Score:", log_grid.best_score_)

# # Final model
# best_C = log_grid.best_params_['classifier__C']

# final_log_model = Pipeline([
#     ('preprocessor', preprocessor),
#     ('classifier', LogisticRegression(
#         C=best_C,
#         max_iter=3000,
#         solver='saga',
#         multi_class='multinomial',
#         class_weight='balanced',
#         random_state=SEED,
#         n_jobs=-1
#     ))
# ])

# final_log_model.fit(X_train, y_train)

# # Evaluation
# log_preds = final_log_model.predict(X_val)
# log_f1 = f1_score(y_val, log_preds, average='macro')

# print(f"Validation Macro F1: {log_f1:.4f}")
# print(classification_report(y_val, log_preds, digits=4))


# final_log_model.fit(X,y)

# test_preds = final_log_model.predict(test)

===== Logistic Regression =====
Best Params: {'classifier__C': 10}
Best CV Score: 0.7953833830692014



Validation Macro F1: 0.7960



              precision |    recall |   f1-score |  support

           0     0.9846    0.9277    0.9553     22835
           1     0.6721    0.8624    0.7555      3183
           2     0.8880    0.8636    0.8757     12488
           3     0.4749    0.8053    0.5975      1094

    accuracy                         0.8989     39600
   macro avg     0.7549    0.8648    0.7960     39600
   
weighted avg     0.9149    0.8989    0.9042     39600

gives : validation marcro f2 score classification report : precision, recall , f1_score , support , it is useful for the imbalenced dataset as, accuracy alone can be misleading.

In [ ]:
# ── Utility ───────────────────────────────────────────────────────────────────
results = {}

def evaluate(name, model, X_val, y_val):
    preds = model.predict(X_val)
    f1    = f1_score(y_val, preds, average='macro')
    results[name] = f1
    print(f"[{name}]  Macro F1 = {f1:.4f}")
    print(classification_report(y_val, preds, digits=4))
    return f1

# LOGISTIC Regression


In [ ]:
log_model = LogisticRegression(
    C=4,
    max_iter=3000,
    class_weight='balanced',
    solver='saga',
    multi_class='multinomial',
    random_state=SEED,
    n_jobs=-1
)

log_model.fit(X_train_t, y_train)
log_f1 = evaluate('Logistic', log_model, X_val_t, y_val)

In [ ]:
# log_model = Pipeline([
#     ('preprocessor', preprocessor),
#     ('classifier', LogisticRegression(
#         C            = 10,       # best params is used : 10
#         max_iter     = 3000,
#         class_weight = 'balanced',
#         solver       = 'saga',    # efficient for large sparse matrices
#         multi_class  = 'multinomial',
#         random_state = SEED,
#         n_jobs       = -1,
#     ))
# ])

# log_model.fit(X_train, y_train)
# log_f1 = evaluate('Logistic', log_model, X_val, y_val)

# print(f"Validation Macro F1 (Logistic Regression): {log_f1:.4f}")

### Model 2 — XGBoost


XGBoost (Extreme Gradient Boosting) is a powerful and efficient machine learning algorithm based on gradient boosting of decision trees. 
- It builds models sequentially, where each new tree corrects the errors of the previous ones, leading to high predictive accuracy.
- widely used because of its speed, scalability, and performance

In [ ]:
# # Auto-detect GPU — prevents crash on CPU-only kernels
# has_gpu      = shutil.which('nvidia-smi') is not None
# xgb_device   = 'cuda' if has_gpu else 'cpu'
# print(f"XGBoost device: {xgb_device}")

# MODEL_NAME = "XGBOOST"

# xgb_pipeline = Pipeline([
#     ('preprocessor', preprocessor),
#     ('classifier', XGBClassifier(
#         tree_method='hist',
#         device=xgb_device,# faster trainng using histogram-based trees       # run model or cpu or gpu
#         eval_metric='mlogloss',    # eval metrics for multi class
#         random_state=SEED,
#         n_jobs=-1                    # all cpu cores
#     ))
# ])

# xgb_param_dist = {
#     'classifier__n_estimators': [300, 500, 700],        # number of trees 
#     'classifier__learning_rate': [0.01, 0.05, 0.1],     # step size
#     'classifier__max_depth': [4, 6, 8],                 # complexity of trees 
# }

# xgb_random = RandomizedSearchCV(
#     xgb_pipeline,
#     xgb_param_dist,
#     n_iter=10,
#     scoring='f1_macro',     # for imbalance multi class problems
#     cv=3,
#     n_jobs=-1,
#     verbose=2,
#     random_state=SEED
# )

# xgb_random.fit(X_train, y_train)

# #  Print best params
# print("\n===== XGBoost =====")
# print("Best Params:", xgb_random.best_params_)
# print("Best CV Score:", xgb_random.best_score_)

# # Final model
# best_xgb = xgb_random.best_params_

# final_xgb_model = Pipeline([
#     ('preprocessor', preprocessor),
#     ('classifier', XGBClassifier(
#         n_estimators=best_xgb['classifier__n_estimators'],
#         learning_rate=best_xgb['classifier__learning_rate'],
#         max_depth=best_xgb['classifier__max_depth'],
#         tree_method='hist',
#         device=xgb_device,
#         eval_metric='mlogloss',
#         random_state=SEED,
#         n_jobs=-1
#     ))
# ])

# final_xgb_model.fit(X_train, y_train)

# # Evaluation
# xgb_preds = final_xgb_model.predict(X_val)
# xgb_f1 = f1_score(y_val, xgb_preds, average='macro')

# print(f"Validation Macro F1: {xgb_f1:.4f}")
# print(classification_report(y_val, xgb_preds, digits=4))

# final_xgb_model.fit(X,y)
# test_preds = final_xgb_model.predict(test)

===== XGBoost =====
Best Params: {'classifier__n_estimators': 700, 'classifier__max_depth': 4, 'classifier__learning_rate': 0.05}
Best CV Score: 0.7865139679923524





Validation Macro F1: 0.7909
2015.4s	1876	              precision    recall  f1-score   support
2015.4s	1877	
2015.4s	1878	           0     0.9770    0.9489    0.9627     22835
2015.4s	1879	           1     0.7813    0.7644    0.7727      3183
2015.4s	1880	           2     0.8404    0.9210    0.8789     12488
2015.4s	1881	           3     0.7585    0.4305    0.5493      1094
2015.4s	1882	
2015.4s	1883	    accuracy                         0.9110     39600
2015.4s	1884	   macro avg     0.8393    0.7662    0.7909     39600
2015.4s	1885	weighted avg     0.9121    0.9110    0.9096     39600


gives : validation marcro f1 score:

classification report : precision, recall , f1_score , support , it is useful for the imbalenced dataset as, accuracy alone can be mislearning.

# XGBOOST CLASSIFIER

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=700,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.85,
    colsample_bytree=0.7,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.05,
    reg_lambda=1.0,
    tree_method='hist',
    eval_metric='mlogloss',
    early_stopping_rounds=40,
    random_state=SEED,
    n_jobs=-1
)

xgb_model.fit(
    X_train_t, y_train,
    eval_set=[(X_val_t, y_val)],
    verbose=100
)

xgb_f1 = evaluate('XGBoost', xgb_model, X_val_t, y_val)

In [ ]:
# # Auto-detect GPU — prevents crash on CPU-only kernels
# has_gpu      = shutil.which('nvidia-smi') is not None
# xgb_device   = 'cuda' if has_gpu else 'cpu'
# print(f"XGBoost device: {xgb_device}")

# xgb_model = Pipeline([
#     ('preprocessor', preprocessor),
#     ('classifier', XGBClassifier(
#         n_estimators          = 700,       # best params: 700
#         learning_rate         = 0.05,      # best params
#         max_depth             = 4,         # best parms
#         subsample             = 0.85,
#         colsample_bytree      = 0.7,    # tightened from 0.9 to reduce overfitting
#         min_child_weight      = 3,
#         gamma                 = 0.1,
#         reg_alpha             = 0.05,
#         reg_lambda            = 1.0,
#         tree_method           = 'hist',
#         device                = xgb_device,
#         eval_metric           = 'mlogloss',
#         early_stopping_rounds = 40,     # constructor (XGB >= 2.0)
#         random_state          = SEED,
#         n_jobs                = -1,
#     ))
# ])

# # Note: eval_set passed to fit for early stopping to work
# # Pipeline exposes fit params via classifier__ prefix
# xgb_model.fit(
#     X_train, y_train,
#     classifier__eval_set = [(
#         xgb_model.named_steps['preprocessor'].fit_transform(X_val), y_val
#     )],
#     classifier__verbose  = 100,
# )
# xgb_f1 = evaluate('XGBoost', xgb_model, X_val, y_val)


# print(f"Validation Macro F1 (XGBoost): {xgb_f1:.4f}")

##  LightGBM Model


LightGBM (Light Gradient Boosting Machine) is a fast and efficient gradient boosting algorithm .
- It builds decision trees using a leaf-wise growth strategy, which makes it faster and often more accurate than traditional boosting methods.
- It is optimized for large-scale and high-dimensional data.

In [ ]:
# import lightgbm as lgb

# def get_lgb_device():
#     try:
#         # try a small dummy model to check GPU
#         test_model = lgb.LGBMClassifier(device='gpu')
#         test_model.fit([[0, 0], [1, 1]], [0, 1])
#         print(" Using GPU for LightGBM")
#         return 'gpu'
#     except:
#         print(" GPU not available, using CPU")
#         return 'cpu'

# lgb_device = get_lgb_device()

In [ ]:
# # ===================== LIGHTGBM PIPELINE =====================

# import lightgbm as lgb
# from lightgbm import LGBMClassifier
# from sklearn.model_selection import RandomizedSearchCV
# from sklearn.metrics import f1_score, classification_report

# # ---------- Step 1: Preprocessing (ONLY ONCE) ----------
# print(" Preprocessing data...")

# X_train_transformed = preprocessor.fit_transform(X_train)
# X_val_transformed = preprocessor.transform(X_val)
# test_transformed = preprocessor.transform(test)

# print(" Preprocessing done!")

# # ---------- Step 2: Base Model ----------
# lgb_model = LGBMClassifier(
#     class_weight='balanced',
#     random_state=SEED,
#     n_jobs=-1,
#     verbose=-1
# )

# # ---------- Step 3: Param Grid ----------
# lgb_param_dist = {
#     'n_estimators': [300, 500],
#     'learning_rate': [0.05, 0.1],
#     'num_leaves': [31, 63],
# }

# # ---------- Step 4: Randomized Search ----------
# lgb_random = RandomizedSearchCV(
#     estimator=lgb_model,
#     param_distributions=lgb_param_dist,
#     n_iter=3,
#     scoring='f1_macro',
#     cv=2,
#     n_jobs=-1,
#     verbose=2,
#     random_state=SEED
# )

# print("\n Running LightGBM tuning...")
# lgb_random.fit(X_train_transformed, y_train)

# print("\n===== LightGBM =====")
# print("Best Params:", lgb_random.best_params_)
# print("Best CV Score:", lgb_random.best_score_)

# # ---------- Step 5: Final Model with Early Stopping ----------
# best_lgb = lgb_random.best_params_

# final_lgb_model = LGBMClassifier(
#     **best_lgb,
#     class_weight='balanced',
#     random_state=SEED,
#     n_jobs=-1,
#     verbose=-1
# )

# print("\n Training final model with early stopping...")
# final_lgb_model.fit(
#     X_train_transformed, y_train,
#     eval_set=[(X_val_transformed, y_val)],
#     eval_metric='multi_logloss',
#     callbacks=[lgb.early_stopping(50, verbose=False)]
# )

# # ---------- Step 6: Evaluation ----------
# lgb_preds = final_lgb_model.predict(X_val_transformed)
# lgb_f1 = f1_score(y_val, lgb_preds, average='macro')

# print(f"\n Validation Macro F1: {lgb_f1:.4f}")
# print(classification_report(y_val, lgb_preds, digits=4))

# # ---------- Step 7: Train on Full Data ----------
# print("\n Training on full dataset...")

# X_full_transformed = preprocessor.fit_transform(X)
# test_transformed = preprocessor.transform(test)

# final_lgb_model.fit(X_full_transformed, y)

# # ---------- Step 8: Test Predictions ----------
# test_preds = final_lgb_model.predict(test_transformed)

# print("\n Prediction distribution:")
# print(pd.Series(test_preds).value_counts().sort_index())

Here is the output of the lightbgm models: 




===== LightGBM =====
Best Params: {'num_leaves': 63, 'n_estimators': 300, 'learning_rate': 0.1}
Best CV Score: 0.8162814741591043



===========================================================

Validation Macro F1: 0.8247

              precision    recall  f1-score   support

           0     0.9823    0.9457    0.9637     22835
           1     0.7297    0.8439    0.7826      3183
           2     0.8801    0.8977    0.8888     12488
           3     0.6355    0.6947    0.6638      1094

    accuracy                         0.9155     39600
   macro avg     0.8069    0.8455    0.8247     39600
weighted avg     0.9202    0.9155    0.9172     39600

- LightGBM uses a leaf-wise tree growth strategy, so controlling num_leaves is critical to prevent overfitting,
-  RandomizedSearchCV tries random combinations of hyperparameters.(fast , no gurantee for absolute best) random pick of parameter

# LIGHTBGM Classifier

In [ ]:
lgb_model = LGBMClassifier(
   n_estimators     = 700,
        learning_rate    = 0.05,
        num_leaves       = 63,       
        subsample        = 0.85,
        colsample_bytree = 0.7,
        reg_alpha        = 0.05,
        reg_lambda       = 0.1,
        min_child_samples= 20,
        class_weight     = 'balanced', 
        random_state     = SEED,
        n_jobs           = -1,
        verbose          = -1
)

lgb_model.fit(X_train_t, y_train)
lgb_f1 = evaluate('LightGBM', lgb_model, X_val_t, y_val)

In [ ]:
# lgb_model = Pipeline([
#     ('preprocessor', preprocessor),
#     ('classifier', LGBMClassifier(
#         n_estimators     = 300,  # best param applied
#         learning_rate    = 0.1,  # best params
#         num_leaves       = 63,        # best params
#         subsample        = 0.85,
#         colsample_bytree = 0.7,
#         reg_alpha        = 0.05,
#         reg_lambda       = 0.1,
#         min_child_samples= 20,
#         class_weight     = 'balanced', 
#         random_state     = SEED,
#         n_jobs           = -1,
#         verbose          = -1,
#     ))
# ])

# lgb_model.fit(X_train, y_train)
# lgb_f1 = evaluate('LightGBM', lgb_model, X_val, y_val)


# print(f"Validation Macro F1 (LightGBM): {lgb_f1:.4f}")

## soft voting ensemble: 




A soft voting ensemble is a technique where multiple models are combined by averaging their predicted probabilities, and the class with the highest combined probability is chosen.


1. This approach reduces the risk of relying on a single model and improves generalization
2.  Different models capture different patterns—linear (Logistic) and non-linear (XGB, LGB). Combining them improves overall performance.

Since I trained three different models—Logistic Regression, XGBoost, and LightGBM—I combined them using a soft voting ensemble. 
- Instead of giving equal importance, I assigned dynamic weights based on their validation F1 scores, so better-performing models contribute more.
- Then I averaged their predicted probabilities and selected the final class using the highest combined probability.”

In [ ]:
import numpy as np

# ---------- Step 1: Weights ----------
total_f1 = log_f1 + xgb_f1 + lgb_f1

w_log = log_f1 / total_f1
w_xgb = xgb_f1 / total_f1
w_lgb = lgb_f1 / total_f1

# optional boost (recommended since LGB is strongest)
w_lgb += 0.05

# normalize again
total = w_log + w_xgb + w_lgb
w_log /= total
w_xgb /= total
w_lgb /= total

print(f"\nWeights → LR: {w_log:.3f} | XGB: {w_xgb:.3f} | LGB: {w_lgb:.3f}")

# ---------- Step 2: Probabilities ----------
log_proba = log_model.predict_proba(X_val_t)
xgb_proba = xgb_model.predict_proba(X_val_t)
lgb_proba = lgb_model.predict_proba(X_val_t)

# ---------- Step 3: Ensemble ----------
ensemble_proba = (
    w_log * log_proba +
    w_xgb * xgb_proba +
    w_lgb * lgb_proba
)

ensemble_pred = np.argmax(ensemble_proba, axis=1)

# ---------- Step 4: Evaluation ----------
ensemble_f1 = f1_score(y_val, ensemble_pred, average='macro')

print(f"\n[Ensemble] Macro F1 = {ensemble_f1:.4f}")
print(classification_report(y_val, ensemble_pred, digits=4))

In [ ]:
# # ── Validation-proportional soft voting ──────────────────────────────

# import numpy as np

# # Step 1:  weight calculation
# total_f1 = log_f1 + xgb_f1 + lgb_f1 + 1e-8   # avoids division error

# w_log = log_f1 / total_f1  # weights assignment is proportional to validation F1_score
# w_xgb = xgb_f1 / total_f1   # rather than equal voting
# w_lgb = lgb_f1 / total_f1

# print(f"\nDynamic weights → LR: {w_log:.3f} | XGB: {w_xgb:.3f} | LGB: {w_lgb:.3f}")

# # Step 2: Use FINAL tuned models for --> probability prediction 
# log_proba = final_log_model.predict_proba(X_val)
# xgb_proba = final_xgb_model.predict_proba(X_val)
# lgb_proba = final_lgb_model.predict_proba(X_val)

# # Step 3: Weighted probability averaging
# ensemble_proba = (
#     w_log * log_proba +
#     w_xgb * xgb_proba +
#     w_lgb * lgb_proba
# )

# # Step 4: Predictions

# # class with the highest combined probability is selected as the final prediction
# ensemble_pred = np.argmax(ensemble_proba, axis=1)

# # Step 5: Evaluation
# ensemble_f1 = f1_score(y_val, ensemble_pred, average='macro')

# print(f"\n[Ensemble] Macro F1 = {ensemble_f1:.4f}")
# print(classification_report(y_val, ensemble_pred, digits=4))

Then , i evaluated the ensemble using macro F1 score to ensure balanced performance across classes

In [ ]:
# # ── Validation-proportional soft voting ──────────────────────────────────────
# total_f1 = log_f1 + xgb_f1 + lgb_f1
# w_log = log_f1 / total_f1
# w_xgb = xgb_f1 / total_f1
# w_lgb = lgb_f1 / total_f1

# print(f"Dynamic weights →  LR: {w_log:.3f}  XGB: {w_xgb:.3f}  LGB: {w_lgb:.3f}")

# log_proba = log_model.predict_proba(X_val)
# xgb_proba = xgb_model.predict_proba(X_val)
# lgb_proba = lgb_model.predict_proba(X_val)

# ensemble_proba = w_log * log_proba + w_xgb * xgb_proba + w_lgb * lgb_proba
# ensemble_pred  = np.argmax(ensemble_proba, axis=1)
# ensemble_f1    = f1_score(y_val, ensemble_pred, average='macro')

# results['Ensemble'] = ensemble_f1
# print(f"\n[Ensemble] Macro F1 = {ensemble_f1:.4f}")
# print(classification_report(y_val, ensemble_pred, digits=4))

##  Model Comparison



In [ ]:
print(f"Logistic Regression F1: {log_f1:.4f}")
print(f"XGBoost F1: {xgb_f1:.4f}")
print(f"LightGBM F1: {lgb_f1:.4f}")
print(f"Ensemble F1: {ensemble_f1:.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Store results
model_names = ["Logistic Regression", "XGBoost", "LightGBM", "Ensemble"]
f1_scores = [log_f1, xgb_f1, lgb_f1, ensemble_f1]

# Plot
plt.figure()
plt.barh(model_names, f1_scores)

plt.xlabel("Macro F1 Score")
plt.title("Model Comparison (Macro F1)")

# Add score labels
for i, v in enumerate(f1_scores):
    plt.text(v, i, f"{v:.4f}")

plt.show()

##  Best Model Selection

In [ ]:
model_scores = {
    'Logistic Regression': log_f1,
    'XGBoost': xgb_f1,
    'LightGBM': lgb_f1,
    'Ensemble': ensemble_f1
}

best_model_name = max(model_scores, key=model_scores.get)

print(f"\nBest model: {best_model_name} (F1 = {model_scores[best_model_name]:.4f})")

##  Final Training on Full Data

Retrain all models on the complete training set (train + val combined).  
The preprocessor is re-fit inside each pipeline — **no leakage** since test labels are never seen.



# TEST PREDICTION

In [ ]:
if best_model_name == 'Ensemble':

    log_proba_test = log_model.predict_proba(test_t)
    xgb_proba_test = xgb_model.predict_proba(test_t)
    lgb_proba_test = lgb_model.predict_proba(test_t)

    test_proba = (
        w_log * log_proba_test +
        w_xgb * xgb_proba_test +
        w_lgb * lgb_proba_test
    )

    test_preds = np.argmax(test_proba, axis=1)

elif best_model_name == 'Logistic Regression':
    test_preds = log_model.predict(test_t)

elif best_model_name == 'XGBoost':
    test_preds = xgb_model.predict(test_t)

else:  # LightGBM
    test_preds = lgb_model.predict(test_t)

print(f"Using: {best_model_name}")
print(pd.Series(test_preds).value_counts().sort_index())

# final submission


In [ ]:
submission = sample_submission.copy()
submission['label'] = test_preds
submission.to_csv('submission.csv', index=False)

print("submission.csv saved!")
print(os.listdir('.'))
submission.head()

# Milestone 1 : 

Q1: What is the shape of the training dataset?

train.shape

Ques 2: how many columns are present in test dataset?

test.shape

Ques 3: How many columns in the training dataset have object data type?

train.info()

Ques 4: Which of the following columns is of boolean type?

Gender,
race,
disability,
emoticon_1

train.head()

Ques 5 : Which of the following columns have missing values? * race religion gender comment

train.isna().sum()

Ques 6: How many distinct target classes are present in the dataset?

train['label'].value_counts()

Ques 7: What percentage of the dataset approximately belongs to label 0

(114173/198000)*100

Ques 8: What is the median number of upvotes per comment?

median_upvotes = train['upvote'].median()
print("Median number of upvotes per comment:", median_upvotes)

Ques 9: Which numerical feature shows the largest maximum value? *

upvote
downvote
if_1
if_2

print(train['upvote'].max())
print(train['downvote'].max())
print(train['if_1'].max())
print(train['if_2'].max())


Question 10: What is the minimum value of if_2?

print('The minimum value of if_2: ', train['if_2'].min())

# Milestone 2:



Q1. In the context of text preprocessing for NLP, which condition must be checked in addition to NaN values when assessing missingness in the comment column of train.csv?

a) Presence of duplicate comments    
b) Presence of out-of-vocabulary tokens     
c) Presence of empty or whitespace-only strings     
d) Presence of rare words     

Sol: Presence of empty or whitespace-only strings


Q2. Convert the created_date column into datetime objects. Identify the month that occurs the most number of times in the dataset and enter the answer as a lowercase string

    sol: train['created_date'] = pd.to_datetime(train['created_date'], errors='coerce')
    train['month'] = train['created_date'].dt.month.fillna(0).astype(int)
    train.month.value_counts()

Q3. Create a new feature called total_emoticons by calculating the sum of emoticon_1, emoticon_2, and emoticon_3 for each row. What is the maximum value observed in this new feature across the entire dataset?

    sol: train['total_emots'] = train[['emoticon_1', 'emoticon_2', 'emoticon_3']].sum(axis=1)
    train['total_emots'].max()

Q4. Calculate the median character length (including spaces) of the comment column for all entries where label is equal to 3. (Note: Treat any missing comments as empty strings).

    sol: train['comment'] = train['comment'].fillna("")
    subset = train[train['label'] == 3]
    subset['char_length'] = subset['comment'].str.len()
    subset['char_length'].median()

Q5. If you apply Min-Max Scaling to the upvote column to normalize it between the range [0, 1], what would be the resulting scaled value for an observation that originally had exactly 10 upvotes?

    sol: x_min = train['upvote'].min()
    x_max = train['upvote'].max()
    (10 - x_min) / (x_max - x_min)

Q6. What is the average word count (number of words separated by whitespace) for comments that are assigned a label of 1? (Round your answer to 2 decimal places).

    sol: train['comment'] = train['comment'].fillna("")
    subset = train[train['label'] == 1]
    subset['word_count'] = subset['comment'].str.split().str.len()
    subset['word_count'].mean()

Q7. How many comments in the entire dataset contain the substring "Trump"? The search should be case-insensitive.

    sol: train['comment'].str.contains("trump", case=False, na=False).sum()

Q8. Take the comment at the very first row (index 0).

- Remove all punctuation.

- Consider the following words as stop words and remove the same :  ['a', 'an', 'the', 'and', 'or', 'but', 'if', 'because', 'as', 'of', 'at', 'by', 'for', 'with', 'about', 'to', 'from', 'up', 'on', 'in', 'out', 'over', 'under', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'it', 'its', 'they', 'them', 'their', 'she', 'her', 'he', 'him', 'his', 'this', 'that', 'which', 'who', 'whom', 'i', 'me', 'my', 'we', 'our', 'you', 'your']
How many words are left in the text after these two operations?

        sol: import string
        
        
        stop_words = set([
            'a','an','the','and','or','but','if','because','as','of','at','by','for','with',
            'about','to','from','up','on','in','out','over','under','is','are','was','were',
            'be','been','being','have','has','had','do','does','did','it','its','they','them',
            'their','she','her','he','him','his','this','that','which','who','whom','i','me',
            'my','we','our','you','your'
        ])
        
        #Get the first comment (index 0), treat missing as empty string
        comment = str(train.loc[0, 'comment'])
        
        # Remove punctuation
        comment_no_punct = comment.translate(str.maketrans('', '', string.punctuation))
        
        # Tokenize by whitespace
        words = comment_no_punct.split()
        
        # Remove stop words (case-insensitive)
        filtered_words = [w for w in words if w.lower() not in stop_words]
        
        # Count remaining words
        len(filtered_words)

Q9. Convert all the text in comment column to lowercase and tokenizing using whitespace. Compute the total number of unique tokens in the dataset and enter the value.

    sol: # Convert to lowercase
    train['comment_lower'] = train['comment'].str.lower()
    
    # Tokenize using whitespace and collect all tokens
    all_tokens = train['comment_lower'].str.split()
    
    # Flatten list of lists into one big list
    from itertools import chain
    flat_tokens = list(chain.from_iterable(all_tokens))
    
    # Get unique tokens
    unique_tokens = set(flat_tokens)
    
    len(unique_tokens)

Q10. Apply the TfidfVectorizer to the comment column of train.csv with stop_words as "english", min_df as 5 and ngram_range as (1,2).

How many TF-IDF features are generated using this configuration? 

    sol: from sklearn.feature_extraction.text import TfidfVectorizer
    
    # Make sure no NaN
    train['comment'] = train['comment'].fillna('')
    
    # Define vectorizer
    tfidf = TfidfVectorizer(
        stop_words='english',
        min_df=5,
        ngram_range=(1,2)
    )
    
    # Fit on comment column
    X_tfidf = tfidf.fit_transform(train['comment'])
    
    # Number of features
    len(tfidf.get_feature_names_out())

# Milestone 3:

#### Milestone 3 


Q1. Split the train dataset using train_test_split with random_state=42 such that 80% is training data and remaining 20% is validation data. store the data as X_train, X_val, y_train, y_val. Lets say training data (X_train) has shape (a,b) where a is number of rows and b is the number of features, similarly validation set (X_val) has shape (c,d) where c is the number of rows and d is the number of features. what will be the value of a + c ?


    sol: from sklearn.model_selection import train_test_split
    
    X = train.drop(columns=['label'])   # Features
    y = train['label']                  # Target
    
    # 80-20 split
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Shapes
    print("X_train shape:", X_train.shape)
    print("X_val shape:", X_val.shape)
    
    # Value of a + c
    a = X_train.shape[0]
    c = X_val.shape[0]
    
    print("a + c =", a + c)


Q2. After converting created_date to a datetime object and extracting the day, month, and year for your X_train and X_val created in the first question, identify the most frequently occurring month across all years in X_train. Which month is it?

    sol: # Convert to datetime
    X_train['created_date'] = pd.to_datetime(X_train['created_date'])
    X_val['created_date'] = pd.to_datetime(X_val['created_date'])
    
    # Extract day, month, year
    X_train['day'] = X_train['created_date'].dt.day
    X_train['month'] = X_train['created_date'].dt.month
    X_train['year'] = X_train['created_date'].dt.year
    
    X_val['day'] = X_val['created_date'].dt.day
    X_val['month'] = X_val['created_date'].dt.month
    X_val['year'] = X_val['created_date'].dt.year
    
    # Most frequent month in X_train
    most_frequent_month = X_train['month'].mode()[0]
    
    print("Most frequent month:", most_frequent_month)

Q3. Impute the null values of categorical features with the value 'none'. Encode only religion', 'gender', 'race' features using one hot encoding, setting handle_unknown='ignore'. Make sure the output is a pandas dataframe. Let's say the shape of X_train after imputing is (a,b), what is the value of b ?
Note: make sure to transform X_val as well using one hot encoding.

    sol: from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder
    
    
    
    # Identify categorical columns
    cat_cols = X_train.select_dtypes(include='object').columns
    
    imputer = SimpleImputer(strategy='constant', fill_value='none')
    
    X_train[cat_cols] = imputer.fit_transform(X_train[cat_cols])
    X_val[cat_cols] = imputer.transform(X_val[cat_cols])
    
    
    
    
    ohe_cols = ['religion', 'gender', 'race']
    
    encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    )
    
    # Fit only on training
    encoded_train = encoder.fit_transform(X_train[ohe_cols])
    encoded_val = encoder.transform(X_val[ohe_cols])

    # Convert to DataFrame
    encoded_cols = encoder.get_feature_names_out(ohe_cols)
    
    encoded_train_df = pd.DataFrame(
        encoded_train,
        columns=encoded_cols,
        index=X_train.index
    )
    
    encoded_val_df = pd.DataFrame(
        encoded_val,
        columns=encoded_cols,
        index=X_val.index
    )
    
    # Drop original encoded columns
    X_train = X_train.drop(columns=ohe_cols)
    X_val = X_val.drop(columns=ohe_cols)
    
    # Concatenate back
    X_train = pd.concat([X_train, encoded_train_df], axis=1)
    X_val = pd.concat([X_val, encoded_val_df], axis=1)
    
    X_train.shape[1]

Q4. Apply CountVectorizer to the column comment of X_train and X_val obtained in previous questions, what is the sum of the counts for the document at index 1 (the second row) of the transformed X_train sparse matrix?
Note: Please ensure to transform X_val as well.

    sol: from sklearn.feature_extraction.text import CountVectorizer
    
    # Initialize CountVectorizer
    vectorizer = CountVectorizer()
    
    # Fit on training data only
    X_train_counts = vectorizer.fit_transform(X_train['comment'])
    
    # Transform validation data
    X_val_counts = vectorizer.transform(X_val['comment'])
    
    # Sum of word counts for document at index 1 (second row)
    X_train_counts[1].sum()

Q5. Convert 'disability' feature into integer type, with True being mapped to 1 and False mapped to 0. What is the sum of all the 'disability values in X_train and X_val after the transformation?

    sol: # Map True -> 1 and False -> 0
    X_train['disability'] = X_train['disability'].map({True: 1, False: 0}).astype(int)
    X_val['disability'] = X_val['disability'].map({True: 1, False: 0}).astype(int)
    
    # Sum of disability values in both datasets
    total_sum = X_train['disability'].sum() + X_val['disability'].sum()
    
    print("Sum of disability values:", total_sum)

Q6. Scale the numeric features using StandardScaler. What are the number of features seen during fit for X_train? Ensure that you drop all datetime columns first.

    sol: from sklearn.preprocessing import StandardScaler
    import numpy as np
    
    # 1️ Drop datetime columns
    datetime_cols = X_train.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns
    
    X_train = X_train.drop(columns=datetime_cols)
    X_val = X_val.drop(columns=datetime_cols)
    
    # 2️ Select numeric columns only
    num_cols = X_train.select_dtypes(include=np.number).columns
    
    #  Scale
    scaler = StandardScaler()
    
    X_train_scaled = scaler.fit_transform(X_train[num_cols])
    X_val_scaled = scaler.transform(X_val[num_cols])
    
    # 4️ Number of features seen during fit
    print("Number of features seen during fit:", scaler.n_features_in_)

Q7.  Preprocess the data in the following manner and train the model:

0) Use the full raw data and first split it into training and validation sets so that 80% of the data forms X_train, y_train and the remaining 20% forms X_val, y_val.

After splitting:

Impute missing values by fitting the SimpleImputer on X_train. You are free to use any valid strategy for Imputation.

Use the fitted imputer to transform both X_train and X_val

Ensure that after imputation no value in X_train or X_val is negative.

If any negative value exists, convert it to positive using the absolute value function

1) Convert the created_date feature into a datetime object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.

2) Transform the "comment" feature using TfidfVectorizer with stop_words='english'. Make sure to transform X_val accordingly.

3) Encode all the categorical features using one hot encoder with handle_unknown='ignore'. Make sure to transform X_val accordingly.
Now train a Multinomial Naive Bayes model on the preprocessed data. Use a pipeline if you need to. What is the macro F1 score obtained on the train dataset?


Note: you may train Multinomial Naive Bayes model on the preprocessed data which has been obtained from previous questions and give the answer, both ways the answers will be accepted. 

If some feature is creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).


    sol: from sklearn.model_selection import train_test_split
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.metrics import f1_score
    from scipy.sparse import hstack, csr_matrix
    
    
    # 0️ SPLIT RAW DATA
    
    
    X = train.drop(columns=['label'])
    y = train['label']
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    
    # 1️ IMPUTATION
    
    
    imputer = SimpleImputer(strategy='most_frequent')
    
    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_val = pd.DataFrame(
        imputer.transform(X_val),
        columns=X_val.columns,
        index=X_val.index
    )
    
    # Ensure no negative values
    X_train = X_train.applymap(lambda x: abs(x) if isinstance(x, (int, float)) else x)
    X_val = X_val.applymap(lambda x: abs(x) if isinstance(x, (int, float)) else x)
    
    
    # 2️ DATE FEATURE ENGINEERING
    
    
    X_train['created_date'] = pd.to_datetime(X_train['created_date'])
    X_val['created_date'] = pd.to_datetime(X_val['created_date'])
    
    for df_ in [X_train, X_val]:
        df_['day'] = df_['created_date'].dt.day
        df_['month'] = df_['created_date'].dt.month
        df_['year'] = df_['created_date'].dt.year
        df_.drop(columns=['created_date'], inplace=True)
    
    
    # 3️ TFIDF ON COMMENT
    
    
    tfidf = TfidfVectorizer(stop_words='english')
    
    X_train_text = tfidf.fit_transform(X_train['comment'])
    X_val_text = tfidf.transform(X_val['comment'])

    X_train.drop(columns=['comment'], inplace=True)
    X_val.drop(columns=['comment'], inplace=True)
    
    
    # 4️ ONE HOT ENCODING (ALL CATEGORICAL)
    
    
    cat_cols = X_train.select_dtypes(include='object').columns
    
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    
    X_train_cat = ohe.fit_transform(X_train[cat_cols])
    X_val_cat = ohe.transform(X_val[cat_cols])
    
    X_train.drop(columns=cat_cols, inplace=True)
    X_val.drop(columns=cat_cols, inplace=True)
    
    
    # 5️ NUMERIC FEATURES
    
    
    X_train_num = csr_matrix(X_train.values)
    X_val_num = csr_matrix(X_val.values)
    
    
    # 6️ COMBINE ALL FEATURES
    
    
    X_train_final = hstack([X_train_num, X_train_cat, X_train_text])
    X_val_final = hstack([X_val_num, X_val_cat, X_val_text])

Q8. What is the macro F1 score obtained on the validation dataset?

    sol: # Predict using fully preprocessed validation data
    y_val_pred = model.predict(X_val_final)
    
    # Compute macro F1
    val_macro_f1 = f1_score(y_val, y_val_pred, average='macro')
    
    print("Macro F1 Score on Validation:", val_macro_f1)

Q9. Process the data in the following manner as given below and train the model.
0) Use the full raw data and first split it into training and validation sets so that 80% of the data forms X_train, y_train and the remaining 20% forms X_val, y_val.
After splitting:

Impute missing values by fitting the SimpleImputer on X_train. You are free to use any valid strategy for Imputation.

Use the fitted imputer to transform both X_train and X_val

Ensure that after imputation no value in X_train or X_val is negative.

If any negative value exists, convert it to positive using the absolute value function

1) Convert the created_date feature into a datetime object and engineer three new numerical features: day, month, and year. keep these three new features and drop the feature created_date. Make sure to transform X_val accordingly.

2) Create a new binary categorical column named is_weekend where Saturday and Sunday are represented as 1 and all other days as 0. 

3) Apply TfidfVectorizer(stop_words='english') to the comment column. Make sure to transform X_val accordingly.

4) Apply OneHotEncoder(handle_unknown='ignore') to the categorical features including newly created categorical feature is_weekend. Make sure to transform X_val accordingly.

After all above transformation, train MultinomialNB model.

After training, use the model to predict the labels for X_train and calculate the Macro F1 score. What is the resulting score rounded to four decimal places?

note: If some features are creating problem or giving error during fit, you can find a way to handle it such as (dropping it, encoding it, changing its data type etc).


    sol: from sklearn.model_selection import train_test_split
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.naive_bayes import MultinomialNB
    from sklearn.metrics import f1_score
    from scipy.sparse import hstack, csr_matrix
    
    
    # 0️ SPLIT RAW DATA
    
    
    X = train.drop(columns=['label'])
    y = train['label']
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    
    # 1️ IMPUTATION
    
    
    imputer = SimpleImputer(strategy='most_frequent')
    
    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_val = pd.DataFrame(
        imputer.transform(X_val),
        columns=X_val.columns,
        index=X_val.index
    )
    
    # Ensure no negative values
    X_train = X_train.applymap(lambda x: abs(x) if isinstance(x, (int, float)) else x)
    X_val = X_val.applymap(lambda x: abs(x) if isinstance(x, (int, float)) else x)
    
    
    # 2️ DATE FEATURE ENGINEERING
    
    
    for df_ in [X_train, X_val]:
        df_['created_date'] = pd.to_datetime(df_['created_date'])
        df_['day'] = df_['created_date'].dt.day
        df_['month'] = df_['created_date'].dt.month
        df_['year'] = df_['created_date'].dt.year
        
        # 3️ CREATE is_weekend
        df_['is_weekend'] = df_['created_date'].dt.weekday.isin([5,6]).astype(int)
        
        df_.drop(columns=['created_date'], inplace=True)
    
    
    # 4️ TFIDF ON COMMENT
    
    
    tfidf = TfidfVectorizer(stop_words='english')
    
    X_train_text = tfidf.fit_transform(X_train['comment'])
    X_val_text = tfidf.transform(X_val['comment'])
    
    X_train.drop(columns=['comment'], inplace=True)
    X_val.drop(columns=['comment'], inplace=True)
    
    
    # 5️ ONE HOT ENCODING (ALL CATEGORICAL INCLUDING is_weekend)
    
    
    cat_cols = X_train.select_dtypes(include='object').columns.tolist()
    
    # Add is_weekend manually if not object
    if 'is_weekend' in X_train.columns:
        cat_cols.append('is_weekend')
    
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    
    X_train_cat = ohe.fit_transform(X_train[cat_cols])
    X_val_cat = ohe.transform(X_val[cat_cols])
    
    X_train.drop(columns=cat_cols, inplace=True)
    X_val.drop(columns=cat_cols, inplace=True)
    
    # 6️ NUMERIC FEATURES
    
    X_train_num = csr_matrix(X_train.values)
    X_val_num = csr_matrix(X_val.values)
    
    # 7️ COMBINE ALL FEATURES
    
    X_train_final = hstack([X_train_num, X_train_cat, X_train_text])
    X_val_final = hstack([X_val_num, X_val_cat, X_val_text])
    
    #8️ TRAIN MODEL
    
    model = MultinomialNB()
    model.fit(X_train_final, y_train)
    
    #9️ MACRO F1 ON TRAIN
    
    y_train_pred = model.predict(X_train_final)
    
    f1_score(y_train, y_train_pred, average='macro')


Q10. What is the macro F1-score on validation set for the previous question's setup?

    sol: y_val_pred = model.predict(X_val_final)
    
    val_macro_f1 = f1_score(y_val, y_val_pred, average='macro')
    
    print(round(val_macro_f1, 4))